# 04 EBM Prompt Enrichment

Retrieve lightweight medical knowledge snippets from EBM-NLP and add them to the prompt as external knowledge.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import EBM_DIR, SEED, SYNTHEA_DIR
from src.data_prep import build_patient_dataset, filter_top_labels
from src.ebm_utils import load_knowledge_snippets, retrieve_knowledge
from src.prompts import prompt_knowledge_ranked
from src.llm_runner import run_openai
from src.parsing import parse_ranked_output

patients_df = pd.read_csv(SYNTHEA_DIR / "patients.csv")
conditions_df = pd.read_csv(SYNTHEA_DIR / "conditions.csv")
df = build_patient_dataset(patients_df, conditions_df)
df, LABEL_SPACE = filter_top_labels(df, top_n=4)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["diagnosis"],
)

knowledge_df = load_knowledge_snippets(EBM_DIR / "knowledge_snippets.csv")
row = test_df.reset_index(drop=True).iloc[0]
snippets = retrieve_knowledge(row["symptoms"], knowledge_df, top_k=2)
prompt = prompt_knowledge_ranked(row, snippets, LABEL_SPACE)
raw_output = run_openai(prompt)
parsed = parse_ranked_output(raw_output, LABEL_SPACE)

snippets, parsed
